[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Saving Changes &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell and the second defines the two models. Run both first.
Each task puts the document back the way it started, so they can be run in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import beanie
import pymongo
from beanie import Document, init_beanie
from pymongo import AsyncMongoClient

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


In [2]:
class Item(Document):
    sku: str
    name: str
    stock: int

    class Settings:
        name = "saving"
        use_state_management = True


class Versioned(Document):
    sku: str
    stock: int

    class Settings:
        name = "saving_versioned"
        use_state_management = True
        use_revision = True


client = AsyncMongoClient(URI)
await init_beanie(database=client.get_default_database(),
                  document_models=[Item, Versioned])
await Item.delete_all()
await Versioned.delete_all()
await Item(sku="A-1", name="a thing", stock=10).insert()
print("ready:", (await Item.find_one(Item.sku == "A-1")).model_dump(exclude={"id"}))


ready: {'sku': 'A-1', 'name': 'a thing', 'stock': 10}


**1.** One field, written back.


In [3]:
item = await Item.find_one(Item.sku == "A-1")
item.stock = 25
await item.save_changes()

print("stored:", (await Item.find_one(Item.sku == "A-1")).stock)


stored: 25


`save_changes` sent a `$set` naming `stock` and nothing else, which is what state management is for.


**2.** What it is about to send.


In [4]:
item = await Item.find_one(Item.sku == "A-1")
print("before touching it:", item.get_changes())

item.stock = 30
print("after a change:    ", item.get_changes())

await item.save_changes()
print("after saving:      ", item.get_changes())


before touching it: {}
after a change:     {'stock': 30}
after saving:       {}


Empty, then the difference, then empty again. The last one is how the object says it is in step with
the database.


**3.** Without reading anything.


In [5]:
await Item.find_one(Item.sku == "A-1").set({Item.name: "set from nowhere"})
print("stored:", (await Item.find_one(Item.sku == "A-1")).name)


stored: set from nowhere


No document was loaded into Python at all. This is the cheapest and safest way to change a field
whose new value you already know.


**4.** Arithmetic on the server.


In [6]:
before = (await Item.find_one(Item.sku == "A-1")).stock
await Item.find_one(Item.sku == "A-1").inc({Item.stock: 7})
after = (await Item.find_one(Item.sku == "A-1")).stock

print("before:", before, "| after:", after)
print("the number was never in Python, so nothing could have raced with it")


before: 30 | after: 37
the number was never in Python, so nothing could have raced with it


Two processes running this at once both land. Reading the value, adding seven and saving would lose
one of them.


**5.** Losing an edit, and keeping it.


In [7]:
for method in ("save", "save_changes"):
    await Item.find_one(Item.sku == "A-1").set({Item.name: "the correct name", Item.stock: 1})

    mine = await Item.find_one(Item.sku == "A-1")
    await Item.find_one(Item.sku == "A-1").set({Item.name: "corrected elsewhere"})

    mine.stock = 50
    await (mine.save() if method == "save" else mine.save_changes())

    after = await Item.find_one(Item.sku == "A-1")
    print(f"  {method:13} stock {after.stock} | name {after.name!r}")


  save          stock 50 | name 'the correct name'
  save_changes  stock 50 | name 'corrected elsewhere'


The same three steps twice. `save` sent every field and put the old name back; `save_changes` sent
only `stock` and left the correction alone.


**6.** Two writers, one document.


In [8]:
await Versioned.delete_all()
await Versioned(sku="V-1", stock=1).insert()

one = await Versioned.find_one(Versioned.sku == "V-1")
two = await Versioned.find_one(Versioned.sku == "V-1")

one.stock = 10
await one.save_changes()
print("the first write:", (await Versioned.find_one(Versioned.sku == "V-1")).stock)

two.stock = 20
try:
    await two.save_changes()
except beanie.exceptions.RevisionIdWasChanged as error:
    print("the second:    ", type(error).__name__, f"message {str(error)!r}")
    print("stored:        ", (await Versioned.find_one(Versioned.sku == "V-1")).stock)

await Item.delete_all()
await Versioned.delete_all()
await client.close()


the first write: 10
the second:     RevisionIdWasChanged message ''
stored:         10


Without `use_revision` the second write would have succeeded and the first would have vanished, with
nothing raised. The exception is the feature.


---

&#8592; **Back to:** [Saving Changes](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/13-saving-changes.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
